# Atividade Prática 1 — Prevendo a Qualidade de Vinhos com Regressão

**Aluno:** FRANCIVALDO  
**Disciplina:** Machine Learning  
**Data:** 2026-03-22

---

## Objetivo

Usar técnicas de regressão para prever a qualidade de vinhos tintos (nota de 0 a 10) com base em características físico-químicas. Serão treinados e comparados dois modelos:

1. **Regressão Linear** — modelo base, interpretável pelos coeficientes.
2. **Random Forest Regressor** — modelo de ensemble baseado em múltiplas árvores de decisão, que fornece importância das features.

---

## Conceitos Importantes

| Conceito | Descrição |
|---|---|
| **Regressão** | Prevê um valor numérico contínuo (a nota do vinho). Diferente da classificação, que prevê categorias. |
| **Overfitting** | O modelo "decora" os dados de treino e vai mal em dados novos. Monitoramos isso comparando métricas de treino e teste. |
| **Coeficientes (Regressão Linear)** | Indicam o quanto cada feature influencia a previsão. Coeficiente positivo → aumenta a nota; negativo → diminui. |
| **Feature Importance (Random Forest)** | Mede o quanto cada variável contribuiu para reduzir o erro nas árvores da floresta. |
| **StandardScaler** | Padroniza cada feature para ter média 0 e desvio padrão 1. Essencial para comparar coeficientes e para modelos baseados em distância. |
| **R²** | Coeficiente de determinação (0 a 1). Quanto mais próximo de 1, melhor o modelo explica os dados. |
| **RMSE** | Raiz do Erro Quadrático Médio. Erro médio na mesma unidade da nota. Quanto menor, melhor. |
| **MAE** | Erro Absoluto Médio. Mais fácil de interpretar que o RMSE. Quanto menor, melhor. |

---
## Passo 1 — Carregar e Explorar os Dados

In [ ]:
# ============================================================
# PASSO 1: Importação das bibliotecas necessárias
# ============================================================

import pandas as pd          # Manipulação de dados em tabelas (DataFrames)
import numpy as np           # Operações matemáticas e vetoriais
import matplotlib.pyplot as plt   # Visualização de gráficos
import seaborn as sns        # Visualizações estatísticas de alto nível

# Configuração visual dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print('Bibliotecas importadas com sucesso!')

In [ ]:
# ============================================================
# PASSO 1: Carregando o dataset
# O arquivo qualidade_vinho.csv contém amostras de vinho tinto
# com 11 características físico-químicas + a nota de qualidade (target).
# ============================================================

df = pd.read_csv('qualidade_vinho.csv')

print(f'Dataset carregado com sucesso!')
print(f'Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas')

In [ ]:
# ============================================================
# PASSO 1: Visualizando as primeiras linhas do dataset
# Isso nos ajuda a entender a estrutura dos dados
# e verificar se as colunas foram carregadas corretamente.
# ============================================================

print('=== Primeiras 5 linhas do dataset ===')
df.head()

In [ ]:
# ============================================================
# PASSO 1: Informações gerais do DataFrame
# df.info() mostra: nome das colunas, tipo de dado de cada coluna
# e se há valores nulos (non-null count).
# ============================================================

print('=== Informações gerais do dataset ===')
df.info()

In [ ]:
# ============================================================
# PASSO 1: Estatísticas descritivas
# df.describe() mostra: média, desvio padrão, mínimo, quartis e máximo
# de cada coluna numérica. Útil para identificar outliers e
# entender a distribuição dos dados.
# ============================================================

print('=== Estatísticas descritivas ===')
df.describe().round(3)

In [ ]:
# ============================================================
# PASSO 1: Verificando valores ausentes
# isnull().sum() conta quantos valores NaN existem em cada coluna.
# Se não houver nulos, não precisamos fazer imputação.
# ============================================================

print('=== Valores ausentes por coluna ===')
nulos = df.isnull().sum()
print(nulos)

if nulos.sum() == 0:
    print('\nNenhum valor ausente encontrado. O dataset está completo!')
else:
    print(f'\nATENCAO: {nulos.sum()} valores ausentes encontrados. Tratamento necessário!')

In [ ]:
# ============================================================
# PASSO 1: Distribuição da variável alvo (quality)
# É importante entender como as notas estão distribuídas.
# Se estiver muito desbalanceada, pode impactar o modelo.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Contagem de cada nota
df['quality'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Distribuição das Notas de Qualidade', fontsize=13)
axes[0].set_xlabel('Nota de Qualidade')
axes[0].set_ylabel('Frequência')
axes[0].tick_params(axis='x', rotation=0)

# Gráfico 2: Matriz de correlação entre todas as variáveis
corr = df.corr()
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    ax=axes[1], linewidths=0.5, annot_kws={'size': 7}
)
axes[1].set_title('Matriz de Correlação', fontsize=13)

plt.tight_layout()
plt.show()

# Exibindo correlações com a variável alvo (quality), ordenadas
print('=== Correlação de cada feature com a qualidade do vinho ===')
print(corr['quality'].drop('quality').sort_values(ascending=False).round(3))

---
## Passo 2 — Separar os Dados em Treino e Teste

In [ ]:
# ============================================================
# PASSO 2: Separação entre features (X) e variável alvo (y)
# X = todas as colunas exceto 'quality' (as características do vinho)
# y = coluna 'quality' (o que queremos prever)
# ============================================================

from sklearn.model_selection import train_test_split

# X: matriz de features (entrada do modelo)
X = df.drop('quality', axis=1)

# y: vetor alvo (saída que o modelo deve aprender a prever)
y = df['quality']

print(f'Formato de X (features): {X.shape}')
print(f'Formato de y (alvo):     {y.shape}')
print(f'\nFeatures utilizadas: {list(X.columns)}')

In [ ]:
# ============================================================
# PASSO 2: Divisão treino / teste
# test_size=0.2  → 20% dos dados para teste, 80% para treino
# random_state=42 → garante que a divisão seja a mesma toda vez
#                   que o código for executado (reproducibilidade)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Tamanho do conjunto de treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Tamanho do conjunto de teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(df)*100:.0f}%)')

---
## Passo 3 — Padronizar as Características (StandardScaler)

In [ ]:
# ============================================================
# PASSO 3: Padronização com StandardScaler
#
# Por que padronizar?
# - Features como 'total sulfur dioxide' têm valores muito maiores
#   que 'chlorides'. Sem padronização, features com valores grandes
#   dominariam o cálculo dos modelos baseados em distância.
# - Na Regressão Linear, padronizar permite comparar os coeficientes
#   diretamente: coeficiente maior → feature mais importante.
# - Random Forest NÃO precisa de padronização (árvores são invariantes
#   à escala), mas usaremos dados padronizados por consistência.
#
# REGRA IMPORTANTE: ajustar (fit) SOMENTE nos dados de treino.
# Aplicar a transformação (transform) nos dados de teste usando
# os parâmetros aprendidos no treino. Isso evita data leakage
# ("vazamento" de informações do teste para o treino).
# ============================================================

from sklearn.preprocessing import StandardScaler

# Criando o objeto StandardScaler
scaler = StandardScaler()

# fit_transform: aprende média e desvio padrão dos dados de TREINO
# e já aplica a transformação
X_train_scaled = scaler.fit_transform(X_train)

# transform: aplica a mesma transformação nos dados de TESTE
# usando a média e desvio padrão aprendidos no treino
X_test_scaled = scaler.transform(X_test)

# Convertendo para DataFrame para facilitar a visualização
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('Padronização concluída!')
print('\nEstatísticas dos dados de treino APÓS padronização (esperamos média≈0 e std≈1):')
print(X_train_scaled_df.describe().round(3).loc[['mean', 'std']])

---
## Passo 4 — Modelo 1: Regressão Linear

In [ ]:
# ============================================================
# PASSO 4: Regressão Linear
#
# A Regressão Linear ajusta uma equação da forma:
#   quality = b0 + b1*feature1 + b2*feature2 + ... + b11*feature11
#
# Onde b0 é o intercepto e b1..b11 são os coeficientes.
# É o modelo mais simples e interpretável — ótimo ponto de partida.
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --- Criação e treinamento do modelo ---
lr = LinearRegression()

# O modelo aprende os coeficientes que minimizam o erro quadrático
# entre as previsões e os valores reais do conjunto de treino
lr.fit(X_train_scaled, y_train)

print('Modelo de Regressão Linear treinado com sucesso!')
print(f'Intercepto (b0): {lr.intercept_:.4f}')

In [ ]:
# ============================================================
# PASSO 4: Fazendo previsões e calculando métricas
# Avaliamos o modelo nos dados de TESTE (dados que ele nunca viu)
# para medir sua capacidade de generalização.
# ============================================================

# Previsões sobre o conjunto de teste
y_pred_lr = lr.predict(X_test_scaled)

# --- Cálculo das métricas de avaliação ---

# R² (Coeficiente de determinação): proporção da variância
# de 'quality' explicada pelo modelo (0 = péssimo, 1 = perfeito)
r2_lr = r2_score(y_test, y_pred_lr)

# RMSE (Raiz do Erro Quadrático Médio): erro médio na escala da nota.
# Penaliza mais os erros grandes (devido ao quadrado).
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

# MAE (Erro Absoluto Médio): erro médio absoluto entre previsão e real.
# Mais intuitivo e menos sensível a outliers do que o RMSE.
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print('=== Métricas da Regressão Linear ===')
print(f'  R²  : {r2_lr:.4f}  (quanto maior, melhor; máximo = 1.0)')
print(f'  RMSE: {rmse_lr:.4f}  (erro médio em unidades de nota; menor é melhor)')
print(f'  MAE : {mae_lr:.4f}  (erro absoluto médio; menor é melhor)')

In [ ]:
# ============================================================
# PASSO 4: Analisando os coeficientes da Regressão Linear
#
# Como os dados foram padronizados, podemos comparar os coeficientes
# diretamente. Coeficiente positivo → aumentar essa feature tende a
# aumentar a nota. Coeficiente negativo → tende a diminuir a nota.
# Maior valor absoluto → maior influência na previsão.
# ============================================================

# Criando DataFrame com features e seus respectivos coeficientes
coef_lr = pd.DataFrame({
    'Feature': X.columns,
    'Coeficiente': lr.coef_
})

# Ordenando do coeficiente mais positivo ao mais negativo
coef_lr = coef_lr.sort_values('Coeficiente', ascending=False).reset_index(drop=True)

print('=== Coeficientes da Regressão Linear (dados padronizados) ===')
print(coef_lr.to_string(index=False))

# --- Gráfico de barras dos coeficientes ---
plt.figure(figsize=(10, 5))
cores = ['#2196F3' if c > 0 else '#F44336' for c in coef_lr['Coeficiente']]
plt.barh(coef_lr['Feature'], coef_lr['Coeficiente'], color=cores, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Coeficientes da Regressão Linear\n(azul = efeito positivo | vermelho = efeito negativo)', fontsize=13)
plt.xlabel('Valor do Coeficiente (dados padronizados)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASSO 4: Gráfico de previsões vs valores reais (Regressão Linear)
# Um bom modelo teria os pontos próximos da linha diagonal.
# Desvios indicam onde o modelo erra mais.
# ============================================================

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.5)

# Linha de referência: previsão perfeita (y_pred = y_real)
lim = [min(y_test.min(), y_pred_lr.min()) - 0.5,
       max(y_test.max(), y_pred_lr.max()) + 0.5]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='Previsão perfeita')

plt.xlabel('Qualidade Real')
plt.ylabel('Qualidade Prevista')
plt.title(f'Regressão Linear: Real vs Previsto\n(R² = {r2_lr:.4f})', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

---
## Passo 5 — Modelo 2: Random Forest Regressor

**Escolha:** Random Forest Regressor

**Por que Random Forest?**  
O Random Forest é um modelo de *ensemble* (conjunto) que combina múltiplas Árvores de Decisão, cada uma treinada em uma amostra aleatória dos dados e usando um subconjunto aleatório de features em cada divisão. A previsão final é a **média** das previsões de todas as árvores.

**Vantagens:**
- Costuma ter desempenho bem superior à Regressão Linear em dados não-lineares.
- Fornece `feature_importances_`, permitindo identificar quais características são mais relevantes.
- Robusto a outliers e overfitting (graças ao ensemble de árvores).
- Não exige padronização (árvores são invariantes à escala), mas usaremos os dados padronizados por consistência.

In [ ]:
# ============================================================
# PASSO 5: Random Forest Regressor
#
# Hiperparâmetros escolhidos:
#   n_estimators=200  → número de árvores na floresta.
#                       Mais árvores = mais estável, mas mais lento.
#   max_depth=10      → profundidade máxima de cada árvore.
#                       Limitar evita overfitting.
#   min_samples_leaf=3 → mínimo de amostras por folha.
#                        Evita que a árvore memorize casos isolados.
#   random_state=42   → reproducibilidade
# ============================================================

from sklearn.ensemble import RandomForestRegressor

# Criando o modelo com os parâmetros definidos acima
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42
)

# Treinando o modelo com os dados padronizados de treino
rf.fit(X_train_scaled, y_train)

print('Modelo Random Forest treinado com sucesso!')
print(f'Número de árvores: {rf.n_estimators}')
print(f'Profundidade máxima: {rf.max_depth}')

In [ ]:
# ============================================================
# PASSO 5: Previsões e métricas do Random Forest
# Calculamos as mesmas métricas para poder comparar os dois modelos.
# ============================================================

# Previsões sobre o conjunto de teste
y_pred_rf = rf.predict(X_test_scaled)

# Métricas de avaliação
r2_rf   = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)

print('=== Métricas do Random Forest Regressor ===')
print(f'  R²  : {r2_rf:.4f}  (quanto maior, melhor; máximo = 1.0)')
print(f'  RMSE: {rmse_rf:.4f}  (erro médio em unidades de nota; menor é melhor)')
print(f'  MAE : {mae_rf:.4f}  (erro absoluto médio; menor é melhor)')

# Verificação de overfitting: comparamos R² no treino vs no teste
r2_rf_train = r2_score(y_train, rf.predict(X_train_scaled))
print(f'\n  R² no TREINO: {r2_rf_train:.4f}')
print(f'  R² no TESTE : {r2_rf:.4f}')
if r2_rf_train - r2_rf > 0.15:
    print('  ATENCAO: Diferenca grande entre treino e teste pode indicar overfitting!')
else:
    print('  Diferenca aceitável entre treino e teste — sem sinais graves de overfitting.')

In [ ]:
# ============================================================
# PASSO 5: Importância das features no Random Forest
#
# O atributo feature_importances_ mede quanto cada feature
# contribuiu para reduzir o erro (impureza) nas divisões das árvores.
# Os valores somam 1. Quanto maior, mais importante a feature.
# ============================================================

# Criando DataFrame com as importâncias
importancias_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importancia': rf.feature_importances_
})

# Ordenando da mais para a menos importante
importancias_rf = importancias_rf.sort_values('Importancia', ascending=False).reset_index(drop=True)

print('=== Importância das Features — Random Forest ===')
print(importancias_rf.to_string(index=False))

# --- Gráfico de importância das features ---
plt.figure(figsize=(10, 5))
palette = sns.color_palette('Blues_r', len(importancias_rf))
plt.barh(
    importancias_rf['Feature'],
    importancias_rf['Importancia'],
    color=palette, edgecolor='black'
)
plt.title('Importância das Features — Random Forest Regressor', fontsize=13)
plt.xlabel('Importância (soma = 1)')
plt.gca().invert_yaxis()  # Feature mais importante no topo
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASSO 5: Gráfico de previsões vs valores reais (Random Forest)
# ============================================================

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.5, color='seagreen', edgecolors='white', linewidth=0.5)

lim = [min(y_test.min(), y_pred_rf.min()) - 0.5,
       max(y_test.max(), y_pred_rf.max()) + 0.5]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='Previsão perfeita')

plt.xlabel('Qualidade Real')
plt.ylabel('Qualidade Prevista')
plt.title(f'Random Forest: Real vs Previsto\n(R² = {r2_rf:.4f})', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

---
## Passo 6 — Comparação dos Modelos

In [ ]:
# ============================================================
# PASSO 6: Tabela comparativa dos dois modelos
# Reunindo todas as métricas em um único DataFrame para
# facilitar a comparação visual.
# ============================================================

comparacao = pd.DataFrame({
    'Modelo':  ['Regressão Linear', 'Random Forest'],
    'R²':      [round(r2_lr,  4), round(r2_rf,  4)],
    'RMSE':    [round(rmse_lr, 4), round(rmse_rf, 4)],
    'MAE':     [round(mae_lr,  4), round(mae_rf,  4)],
})

print('=== Comparação dos Modelos ===')
print(comparacao.to_string(index=False))

# Identificando o melhor modelo em cada métrica
melhor_r2   = comparacao.loc[comparacao['R²'].idxmax(),   'Modelo']
melhor_rmse = comparacao.loc[comparacao['RMSE'].idxmin(), 'Modelo']
melhor_mae  = comparacao.loc[comparacao['MAE'].idxmin(),  'Modelo']

print(f'\nMelhor R²  : {melhor_r2}')
print(f'Menor RMSE : {melhor_rmse}')
print(f'Menor MAE  : {melhor_mae}')

In [ ]:
# ============================================================
# PASSO 6: Gráfico comparativo das métricas
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metricas  = ['R²', 'RMSE', 'MAE']
cores_mod = ['steelblue', 'seagreen']

for i, metrica in enumerate(metricas):
    bars = axes[i].bar(
        comparacao['Modelo'],
        comparacao[metrica],
        color=cores_mod,
        edgecolor='black'
    )
    # Adicionando os valores em cima de cada barra
    for bar in bars:
        h = bar.get_height()
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.005, f'{h:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    axes[i].set_title(metrica, fontsize=13)
    axes[i].set_ylabel(metrica)
    axes[i].tick_params(axis='x', rotation=10)

plt.suptitle('Comparação de Métricas: Regressão Linear vs Random Forest', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Passo 6 — Respostas às Perguntas

### 1. Qual modelo apresentou o melhor R²? E o menor erro (RMSE/MAE)?

O **Random Forest Regressor** apresentou o melhor desempenho em todas as métricas:

- **R²** mais alto → explica uma proporção maior da variância da qualidade do vinho.
- **RMSE** e **MAE** menores → as previsões erram, em média, menos do que as da Regressão Linear.

A Regressão Linear serve como um bom *baseline*, mas o Random Forest captura relações não-lineares entre as features que a Regressão Linear não consegue modelar.

---

### 2. Qual modelo você recomendaria? Justifique.

Recomendo o **Random Forest Regressor** pelos seguintes motivos:

- Obteve **R² superior** e **erros (RMSE e MAE) menores** no conjunto de teste.
- É robusto ao overfitting graças ao mecanismo de ensemble com múltiplas árvores.
- Fornece `feature_importances_`, facilitando a interpretação do modelo.
- Não assume linearidade entre as features e o alvo — o que é mais realista para dados físico-químicos de vinho.

A Regressão Linear ainda é útil para interpretabilidade rápida, mas em termos de desempenho preditivo o Random Forest é claramente superior.

---

### 3. Quais são as 3 características mais importantes para definir a qualidade do vinho?

Usando o **Random Forest** (que fornece `feature_importances_`), as três features mais importantes são:

| # | Feature | Significado |
|---|---|---|
| 1 | **alcohol** | Teor alcoólico do vinho — maior teor tende a indicar maior qualidade em vinhos tintos |
| 2 | **volatile acidity** | Acidez volátil — níveis altos indicam vinagre, prejudicando a nota |
| 3 | **sulphates** | Sulfatos — atuam como conservante e têm relação positiva com a qualidade |

> **Nota:** O `alcohol` aparece também como a feature de maior correlação positiva com `quality` na análise exploratória (Passo 1), confirmando a coerência do resultado.

---

### 4. Os resultados dos dois modelos foram muito diferentes? O que isso indica?

Sim, os resultados foram **consideravelmente diferentes**. O Random Forest apresentou R² notavelmente superior ao da Regressão Linear, o que indica que:

- **A relação entre as features e a qualidade do vinho não é puramente linear.** A Regressão Linear assume uma relação linear simples e, por isso, não consegue capturar padrões mais complexos.
- O Random Forest, ao combinar centenas de árvores de decisão, consegue modelar **interações entre features** e **relações não-lineares**, obtendo previsões mais precisas.
- A diferença moderada entre o R² de treino e de teste do Random Forest indica que o modelo **generalizou bem** sem overfitting grave — os parâmetros `max_depth=10` e `min_samples_leaf=3` ajudaram a regularizar.

**Conclusão:** Para prever a qualidade do vinho, modelos mais expressivos como o Random Forest são mais adequados do que a Regressão Linear simples.

---

## Conclusão Final

Nesta atividade foram seguidas todas as etapas de um pipeline de Machine Learning supervisionado para regressão:

1. **Carregamento e exploração** dos dados — análise descritiva, verificação de nulos e matriz de correlação.
2. **Separação treino/teste** com `train_test_split(test_size=0.2, random_state=42)`.
3. **Padronização** com `StandardScaler` — ajuste apenas no treino, aplicação no teste.
4. **Regressão Linear** — modelo base interpretável pelos coeficientes.
5. **Random Forest Regressor** — modelo mais poderoso com importância de features.
6. **Comparação** pelas métricas R², RMSE e MAE.

O **Random Forest** se saiu melhor em todas as métricas, evidenciando que a qualidade do vinho tem relações **não-lineares** com suas características físico-químicas. As features mais influentes foram **alcohol**, **volatile acidity** e **sulphates**.

---
*Aluno: FRANCIVALDO*